# 📊 Аналитический отчёт по продажам
> Полный анализ выручки, маржи, ABC-анализ, лояльность и market basket — по всем складам и по каждому отдельно.

**Содержание:**
1. [Настройки](#настройки)
2. [Загрузка данных](#загрузка)
3. [KPI-плашки](#kpi)
4. [Динамика выручки и маржи](#динамика)
5. [Сравнение складов](#склады)
6. [Категорийный анализ](#категории)
7. [ABC-анализ товаров](#abc)
8. [Программа лояльности](#лояльность)
9. [Market Basket Analysis](#basket)
10. [Тепловая карта выручки по дням недели](#хeatmap)
11. [Распределение чеков по позициям](#check_dist)
12. [Матрица BCG](#bcg)
13. [Тепловая карта среднего чека](#heatmap_avg)
14. [Уникальность ассортимента по складам](#unique_dishes)
15. [Анализ кассиров](#cashiers)
16. [Почасовой анализ трафика](#hourly)
17. [ABC-анализ по марже](#abc_margin)
18. [Себестоимость во времени](#cost_dynamics)
19. [Итоговое резюме](#резюме)


## ⚙️ 0. Импорт библиотек и подключение

In [ ]:
# ============================================
# ШАГ 1: ИМПОРТ БИБЛИОТЕК И ПОДКЛЮЧЕНИЕ К API
# ============================================

import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from businessru_api import BusinessruAPI
from dotenv import load_dotenv

# Загружаем ключи из .env файла
load_dotenv()
ACCOUNT = os.getenv('BR_ACCOUNT')
APP_ID = os.getenv('BR_APP_ID')
SECRET = os.getenv('BR_SECRET')

# Проверяем, что ключи загружены
print(f"Аккаунт: {ACCOUNT}")
print(f"App ID: {APP_ID[:5]}...")  # показываем только первые 5 символов для безопасности

# Инициализация клиента (токен получается автоматически)
client = BusinessruAPI(ACCOUNT, APP_ID, SECRET)
print("Подключение к Бизнес.Ру установлено")

## ⚙️ 0. Настройки отчёта <a id='настройки'></a>

> Измените только эту ячейку — всё остальное подстроится автоматически.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║              НАСТРОЙКИ ОТЧЁТА — ТОЛЬКО ЗДЕСЬ                   ║
# ╚══════════════════════════════════════════════════════════════════╝

FILE_PATH   = 'data/sales.csv'   # путь к файлу с данными
DATE_FROM   = '2026-03-09'  # None = самая ранняя дата в файле, или '2024-01-01'
DATE_TO     = '2026-05-03'  # None = самая поздняя дата в файле, или '2024-03-31'
TOP_N       = 15                  # топ-N товаров в графиках
MIN_SUPPORT = 0.005               # минимальная поддержка для basket analysis

## 📦 1. Импорты и вспомогательные функции <a id='загрузка'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from itertools import combinations
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ── Стиль ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.facecolor':  '#F8F9FA',
    'axes.facecolor':    '#F8F9FA',
    'axes.grid':         True,
    'grid.alpha':        0.4,
    'grid.linestyle':    '--',
})

PALETTE = [
    '#2563EB','#10B981','#F59E0B','#EF4444',
    '#8B5CF6','#EC4899','#06B6D4','#84CC16',
    '#F97316','#6366F1','#14B8A6','#FBBF24',
]

def fmt_rub(x, pos=None):
    if x >= 1_000_000: return f'{x/1_000_000:.1f}М'
    if x >= 1_000:     return f'{x/1_000:.0f}К'
    return f'{x:.0f}'

def rub(v):
    return f'{v:,.0f} ₽'.replace(',', ' ')

def pct(v):
    return f'{v:.1f}%'

print('✅ Библиотеки загружены')



## 📂 2. Загрузка и подготовка данных

In [ ]:
import pandas as pd
import csv

encoding = 'utf-8'  # или 'cp1251'

# ── Чтение файла ──────────────────────────────────────────────────
# Проверяем, не склеены ли колонки в одну
with open(FILE_PATH, 'r', encoding=encoding) as f:
    first_line = f.readline().strip()
    if ',' in first_line:
        # Нормальный CSV с разделителями
        df_raw = pd.read_csv(
            FILE_PATH,
            sep=',',
            encoding=encoding,
            quotechar='"',
            quoting=csv.QUOTE_MINIMAL,
            on_bad_lines='warn',
            engine='python',
        )
    else:
        # Склеенные колонки - нужно определить разделитель
        print("⚠️ Обнаружены возможные проблемы с разделителями. Пробуем определить...")
        df_raw = pd.read_csv(
            FILE_PATH,
            sep=None,  # автоопределение разделителя
            encoding=encoding,
            engine='python',
        )

# ── Посмотрим, какие колонки реально загрузились ─────────────────
print("Доступные колонки:")
print(df_raw.columns.tolist())

# ── Переименование / нормализация колонок ─────────────────────────
df_raw.columns = df_raw.columns.str.strip()

print("\nКолонки после очистки:")
print(df_raw.columns.tolist())

# Если все еще одна колонка - пробуем разделить вручную
if len(df_raw.columns) == 1:
    print("\n⚠️ Обнаружены склеенные колонки. Разделяем...")
    
    # Пробуем прочитать файл с другим разделителем
    df_raw = pd.read_csv(
        FILE_PATH,
        sep=';',  # пробуем точку с запятой
        encoding=encoding,
        quotechar='"',
        on_bad_lines='warn',
    )
    
    if len(df_raw.columns) == 1:
        # Если все еще одна колонка, пробуем табуляцию
        df_raw = pd.read_csv(
            FILE_PATH,
            sep='\t',
            encoding=encoding,
            on_bad_lines='warn',
        )
    
    print(f"\nКолонок после исправления: {len(df_raw.columns)}")
    print(df_raw.columns.tolist())

# ── Даты ──────────────────────────────────────────────────────────
df_raw['Учетный день'] = pd.to_datetime(df_raw['Учетный день'], dayfirst=True, errors='coerce')

_date_from = pd.to_datetime(DATE_FROM) if DATE_FROM else df_raw['Учетный день'].min()
_date_to   = pd.to_datetime(DATE_TO)   if DATE_TO   else df_raw['Учетный день'].max()

df = df_raw[
    (df_raw['Учетный день'] >= _date_from) &
    (df_raw['Учетный день'] <= _date_to)
].copy()

# ── Числовые колонки ──────────────────────────────────────────────
for col in ['Сумма со скидкой, р.', 'Сумма без скидки, р.', 'Скидка, р.']:
    if col in df.columns:
        df[col] = (df[col].astype(str)
                          .str.replace(' ', '', regex=False)
                          .str.replace(',', '.', regex=False)
                          .pipe(pd.to_numeric, errors='coerce')
                          .fillna(0))

# ── Ключ чека ─────────────────────────────────────────────────────
if 'Номер чека' in df.columns and 'Со склада' in df.columns:
    df['check_key'] = df['Со склада'].astype(str) + '_' + df['Номер чека'].astype(str)
elif 'Номер чека' in df.columns:
    df['check_key'] = df['Номер чека'].astype(str)
else:
    df['check_key'] = df.index.astype(str)

# ── Склады ────────────────────────────────────────────────────────
warehouses = df['Со склада'].dropna().unique().tolist() if 'Со склада' in df.columns else []
if 'Со склада' in df.columns:
    df['Склад'] = df['Со склада']

# ── Себестоимость ─────────────────────────────────────────────────
cost_candidates = ['Себестоимость, р.', 'Себест., р.', 'Себестоимость']
cost_col = next((c for c in cost_candidates if c in df.columns), None)
has_cost = cost_col is not None

if has_cost:
    df[cost_col] = (df[cost_col].astype(str)
                                .str.replace(' ', '', regex=False)
                                .str.replace(',', '.', regex=False)
                                .pipe(pd.to_numeric, errors='coerce')
                                .fillna(0))
    df['Маржа, р.'] = df['Сумма со скидкой, р.'] - df[cost_col]

# ── Кассир ────────────────────────────────────────────────────────
cashier_candidates = ['Кассир', 'Сотрудник', 'Продавец']
cashier_col = next((c for c in cashier_candidates if c in df.columns), None)
has_cashier = cashier_col is not None

# ── Время открытия ────────────────────────────────────────────────
has_time = 'Время открытия' in df.columns
if has_time:
    df['Час открытия'] = pd.to_datetime(
        df['Время открытия'], errors='coerce'
    ).dt.hour

# ── Неделя (для анализа себестоимости во времени) ─────────────────
df['Неделя'] = df['Учетный день'] - pd.to_timedelta(
    df['Учетный день'].dt.dayofweek, unit='d'
)

# ── Совместимость: недостающие колонки ───────────────────────────
if 'Скидка, р.' not in df.columns and 'Сумма без скидки, р.' in df.columns and 'Сумма со скидкой, р.' in df.columns:
    df['Скидка, р.'] = (df['Сумма без скидки, р.'] - df['Сумма со скидкой, р.']).clip(lower=0)

if 'Тип скидки' not in df.columns:
    df['Тип скидки'] = None

if 'Номер карты клиента' not in df.columns:
    df['Номер карты клиента'] = None

# ── Ключ чека (обновленный с учетом даты) ─────────────────────────
if 'Со склада' in df.columns and 'Номер чека' in df.columns:
    df['check_key'] = (
        df['Со склада'].astype(str) + '_' +
        df['Учетный день'].dt.strftime('%Y%m%d') + '_' +
        df['Номер чека'].astype(str)
    )

print(f'✅ Загружено строк: {len(df):,}'.replace(',', ' '))
print(f'   Период:   {_date_from.date()} — {_date_to.date()}')
if warehouses:
    print(f'   Склады:   {warehouses}')
print(f'   Маржа:    {"есть → " + cost_col if has_cost else "нет"}')
print(f'   Кассир:   {"есть → " + cashier_col if has_cashier else "нет"}')
print(f'   Время:    {"есть" if has_time else "нет"}')
print(f'\n   Колонки: {list(df.columns)}')

## 💰 3. KPI-плашки — сводные показатели <a id='kpi'></a>

In [ ]:
if len(df) == 0:
    raise ValueError(f'⛔ Нет данных за период {_date_from} — {_date_to}. Проверьте даты в настройках.')

total_rev    = df['Сумма со скидкой, р.'].sum()
total_cost   = df[cost_col].sum() if has_cost else None
total_margin = df['Маржа, р.'].sum() if has_cost else None
rev_base     = total_rev if total_rev != 0 else 1
margin_pct   = total_margin / rev_base * 100 if has_cost else None
n_checks     = df['check_key'].nunique()
avg_check    = total_rev / n_checks if n_checks > 0 else 0
total_disc   = df['Скидка, р.'].sum()
rev_no_disc  = df['Сумма без скидки, р.'].sum()
disc_pct     = total_disc / rev_no_disc * 100 if rev_no_disc != 0 else 0
bonus_checks = df[df['Тип скидки'] == 'Бонусы']['check_key'].nunique()
bonus_pct    = bonus_checks / n_checks * 100 if n_checks > 0 else 0

kpis = [
    ('💰 Выручка',         rub(total_rev),         '#2563EB'),
    ('📈 Маржа',           f'{rub(total_margin)}\n{pct(margin_pct)}' if has_cost else 'Нет данных', '#10B981'),
    ('🧾 Чеков',           f'{n_checks:,}'.replace(',', ' '),         '#F59E0B'),
    ('🛒 Средний чек',     rub(avg_check),          '#8B5CF6'),
    ('🏷 Скидки',          f'{rub(total_disc)}\n{pct(disc_pct)}',      '#EF4444'),
    ('🎁 Чеков с бонусами',f'{bonus_checks:,} ({pct(bonus_pct)})'.replace(',', ' '), '#EC4899'),
]

fig, axes = plt.subplots(1, len(kpis), figsize=(18, 2.4))
fig.patch.set_facecolor('#1E293B')

for ax, (label, value, color) in zip(axes, kpis):
    ax.set_facecolor('#1E293B')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')
    rect = mpatches.FancyBboxPatch((0.05, 0.05), 0.90, 0.90,
        boxstyle='round,pad=0.02', linewidth=2,
        edgecolor=color, facecolor='#0F172A')
    ax.add_patch(rect)
    ax.text(0.5, 0.72, label,  ha='center', va='center', fontsize=9.5,
            color='#94A3B8', fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.35, value,  ha='center', va='center', fontsize=12.5,
            color=color,     fontweight='bold', transform=ax.transAxes,
            linespacing=1.4)

period_str = f'{_date_from}  —  {_date_to}'
fig.suptitle(f'Сводный отчёт по продажам · {period_str}',
             fontsize=14, fontweight='bold', color='white', y=1.08)
plt.tight_layout()
plt.show()

# ── KPI-плашки по каждому складу ──────────────────────────────────────────
print('\n📦 KPI по складам:')
for wh in warehouses:
    df_wh = df[df['Склад'] == wh]
    if len(df_wh) == 0:
        continue
    wh_rev    = df_wh['Сумма со скидкой, р.'].sum()
    wh_cost   = df_wh[cost_col].sum() if has_cost else None
    wh_margin = df_wh['Маржа, р.'].sum() if has_cost else None
    wh_rev_base = wh_rev if wh_rev != 0 else 1
    wh_margin_pct = wh_margin / wh_rev_base * 100 if has_cost else None
    wh_checks = df_wh['check_key'].nunique()
    wh_avg    = wh_rev / wh_checks if wh_checks > 0 else 0
    wh_disc   = df_wh['Скидка, р.'].sum()
    wh_rev_nd = df_wh['Сумма без скидки, р.'].sum()
    wh_disc_pct = wh_disc / wh_rev_nd * 100 if wh_rev_nd != 0 else 0
    wh_bonus_checks = df_wh[df_wh['Тип скидки'] == 'Бонусы']['check_key'].nunique()
    wh_bonus_pct = wh_bonus_checks / wh_checks * 100 if wh_checks > 0 else 0

    wh_kpis = [
        ('💰 Выручка',         rub(wh_rev),         '#2563EB'),
        ('📈 Маржа',           f'{rub(wh_margin)}\n{pct(wh_margin_pct)}' if has_cost else 'Нет данных', '#10B981'),
        ('🧾 Чеков',           f'{wh_checks:,}'.replace(',', ' '),         '#F59E0B'),
        ('🛒 Средний чек',     rub(wh_avg),          '#8B5CF6'),
        ('🏷 Скидки',          f'{rub(wh_disc)}\n{pct(wh_disc_pct)}',      '#EF4444'),
        ('🎁 Чеков с бонусами',f'{wh_bonus_checks:,} ({pct(wh_bonus_pct)})'.replace(',', ' '), '#EC4899'),
    ]

    fig_wh, axes_wh = plt.subplots(1, len(wh_kpis), figsize=(18, 2.4))
    fig_wh.patch.set_facecolor('#1E293B')

    for ax_wh, (label_wh, value_wh, color_wh) in zip(axes_wh, wh_kpis):
        ax_wh.set_facecolor('#1E293B')
        ax_wh.set_xlim(0, 1); ax_wh.set_ylim(0, 1)
        ax_wh.axis('off')
        rect_wh = mpatches.FancyBboxPatch((0.05, 0.05), 0.90, 0.90,
            boxstyle='round,pad=0.02', linewidth=2,
            edgecolor=color_wh, facecolor='#0F172A')
        ax_wh.add_patch(rect_wh)
        ax_wh.text(0.5, 0.72, label_wh,  ha='center', va='center', fontsize=9.5,
                color='#94A3B8', fontweight='bold', transform=ax_wh.transAxes)
        ax_wh.text(0.5, 0.35, value_wh,  ha='center', va='center', fontsize=12.5,
                color=color_wh,  fontweight='bold', transform=ax_wh.transAxes,
                linespacing=1.4)

    fig_wh.suptitle(f'📦 {wh} · {_date_from}  —  {_date_to}',
                 fontsize=14, fontweight='bold', color='white', y=1.08)
    plt.tight_layout()
    plt.show()


## 📈 4. Динамика выручки и маржи <a id='динамика'></a>

Общая динамика по всем складам, затем — по каждому складу отдельно.

In [ ]:
def plot_revenue_dynamics(data, title_prefix=''):
    daily = data.groupby('Учетный день').agg(
        Выручка=('Сумма со скидкой, р.', 'sum'),
        **({'Маржа': ('Маржа, р.', 'sum')} if has_cost else {})
    ).reset_index()
    daily['MA7'] = daily['Выручка'].rolling(7, min_periods=1).mean()

    fig, axes = plt.subplots(2 if has_cost else 1, 1,
                              figsize=(16, 9 if has_cost else 5), sharex=True)
    if not has_cost:
        axes = [axes]

    ax = axes[0]
    ax.bar(daily['Учетный день'], daily['Выручка'], color=PALETTE[0], alpha=0.45, width=0.9, label='Выручка по дням')
    ax.plot(daily['Учетный день'], daily['MA7'], color=PALETTE[0], linewidth=2.5, label='Скольз. среднее (7 дн.)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
    ax.set_title(f'{title_prefix}Динамика выручки')
    ax.legend(frameon=False)
    ax.set_ylabel('Выручка, ₽')

    if has_cost:
        ax2 = axes[1]
        daily['MA7_margin'] = daily['Маржа'].rolling(7, min_periods=1).mean()
        ax2.bar(daily['Учетный день'], daily['Маржа'], color=PALETTE[1], alpha=0.45, width=0.9, label='Маржа по дням')
        ax2.plot(daily['Учетный день'], daily['MA7_margin'], color=PALETTE[1], linewidth=2.5, label='Скольз. среднее (7 дн.)')
        ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
        ax2.set_title(f'{title_prefix}Динамика маржи')
        ax2.legend(frameon=False)
        ax2.set_ylabel('Маржа, ₽')

    plt.tight_layout()
    plt.show()

### 🌐 Все склады

In [ ]:
plot_revenue_dynamics(df, title_prefix='[Все склады] ')

### 🏭 По каждому складу

In [ ]:
for wh in warehouses:
    df_wh = df[df['Склад'] == wh]
    if len(df_wh) == 0:
        continue
    print(f'\n📦 Склад: {wh}')
    plot_revenue_dynamics(df_wh, title_prefix=f'[{wh}] ')

## 🏭 5. Сравнение складов <a id='склады'></a>

In [ ]:
wh_agg = df.groupby('Склад').agg(
    Выручка=('Сумма со скидкой, р.', 'sum'),
    Чеков=('check_key', 'nunique'),
    **({'Себестоимость': (cost_col, 'sum')} if has_cost else {})
).reset_index()
wh_agg['Средний чек'] = wh_agg['Выручка'] / wh_agg['Чеков']
if has_cost:
    wh_agg['Маржа'] = wh_agg['Выручка'] - wh_agg['Себестоимость']
    wh_agg['Маржа, %'] = wh_agg['Маржа'] / wh_agg['Выручка'] * 100
wh_agg = wh_agg.sort_values('Выручка', ascending=False)

ncols = 3 if has_cost else 2
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))
fig.suptitle('Сравнение складов', fontsize=14, fontweight='bold')
colors = [PALETTE[i] for i in range(len(wh_agg))]

axes[0].barh(wh_agg['Склад'], wh_agg['Выручка'], color=colors)
for i, v in enumerate(wh_agg['Выручка']):
    axes[0].text(v * 0.02, i, rub(v), va='center', fontsize=9, fontweight='bold')
axes[0].set_title('Выручка')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
axes[0].invert_yaxis()

axes[1].barh(wh_agg['Склад'], wh_agg['Средний чек'], color=colors)
for i, v in enumerate(wh_agg['Средний чек']):
    axes[1].text(v * 0.02, i, rub(v), va='center', fontsize=9, fontweight='bold')
axes[1].set_title('Средний чек')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
axes[1].invert_yaxis()

if has_cost:
    axes[2].barh(wh_agg['Склад'], wh_agg['Маржа, %'], color=colors)
    for i, v in enumerate(wh_agg['Маржа, %']):
        axes[2].text(v * 0.02, i, pct(v), va='center', fontsize=9, fontweight='bold')
    axes[2].set_title('Маржинальность')
    axes[2].set_xlabel('%')
    axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

print('\n📊 Детальная таблица по складам:')
display_cols = ['Склад', 'Выручка', 'Чеков', 'Средний чек']
if has_cost:
    display_cols += ['Маржа', 'Маржа, %']
styled = wh_agg[display_cols].copy()
styled['Выручка'] = styled['Выручка'].apply(rub)
styled['Средний чек'] = styled['Средний чек'].apply(rub)
if has_cost:
    styled['Маржа'] = styled['Маржа'].apply(rub)
    styled['Маржа, %'] = styled['Маржа, %'].apply(pct)
print(styled.to_string(index=False))

## 📂 6. Категорийный анализ <a id='категории'></a>

Анализ по категориям блюда и бухгалтерским категориям — общий и по каждому складу.

In [ ]:
def plot_category(data, cat_col, title):
    agg = data.groupby(cat_col).agg(
        Выручка=('Сумма со скидкой, р.', 'sum'),
        Продаж=('Блюдо', 'count'),
        **({'Маржа': ('Маржа, р.', 'sum')} if has_cost else {})
    ).reset_index().sort_values('Выручка', ascending=True)

    if len(agg) == 0:
        print(f'   ⚠️ Нет данных для: {title}')
        return

    ncols = 3 if has_cost else 2
    fig, axes = plt.subplots(1, ncols, figsize=(5.5 * ncols, max(4, len(agg) * 0.5 + 1.5)))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(agg))]

    axes[0].barh(agg[cat_col], agg['Выручка'], color=colors)
    for i, v in enumerate(agg['Выручка']):
        axes[0].text(v * 0.01, i, rub(v), va='center', fontsize=8.5)
    axes[0].set_title('Выручка')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))

    axes[1].barh(agg[cat_col], agg['Продаж'], color=colors, alpha=0.7)
    for i, v in enumerate(agg['Продаж']):
        axes[1].text(v * 0.01, i, f'{v:,}'.replace(',', ' '), va='center', fontsize=8.5)
    axes[1].set_title('Количество продаж')

    if has_cost:
        agg['Маржа, %'] = agg['Маржа'] / agg['Выручка'] * 100
        bar_colors = [PALETTE[1] if v >= 40 else PALETTE[3] for v in agg['Маржа, %']]
        axes[2].barh(agg[cat_col], agg['Маржа, %'], color=bar_colors)
        for i, v in enumerate(agg['Маржа, %']):
            axes[2].text(v * 0.01, i, pct(v), va='center', fontsize=8.5)
        axes[2].set_title('Маржинальность (%)')
        axes[2].axvline(40, color='gray', linestyle='--', alpha=0.6, label='Порог 40%')
        axes[2].legend(frameon=False, fontsize=8)

    plt.tight_layout()
    plt.show()

### 🌐 Все склады — категории блюда

In [ ]:
plot_category(df, 'Категория блюда', '📂 [Все склады] Анализ по категориям блюда')

### 🌐 Все склады — бухгалтерские категории

In [ ]:
plot_category(df, 'Бухгалтерская категория блюд', '📂 [Все склады] Анализ по бухгалтерским категориям')

### 🏭 По каждому складу

In [ ]:
for wh in warehouses:
    df_wh = df[df['Склад'] == wh]
    if len(df_wh) == 0:
        continue
    print(f'\n📦 Склад: {wh}')
    plot_category(df_wh, 'Категория блюда',         f'📂 [{wh}] Анализ по категориям блюда')
    plot_category(df_wh, 'Бухгалтерская категория блюд', f'📂 [{wh}] Анализ по бухгалтерским категориям')

## 🏆 7. ABC-анализ товаров — Топ-15 блюд <a id='abc'></a>

- **Класс A** — 80% выручки
- **Класс B** — следующие 15% (до 95%)
- **Класс C** — оставшиеся 5%

In [ ]:
def plot_abc(data, title_prefix=''):
    dish_agg = data.groupby('Блюдо').agg(
        Выручка=('Сумма со скидкой, р.', 'sum'),
        Продаж=('Блюдо', 'count'),
        **({'Маржа': ('Маржа, р.', 'sum')} if has_cost else {})
    ).reset_index()

    if len(dish_agg) == 0:
        return dish_agg

    dish_agg = dish_agg.sort_values('Выручка', ascending=False).reset_index(drop=True)
    dish_agg['Доля выручки, %'] = dish_agg['Выручка'] / dish_agg['Выручка'].sum() * 100
    dish_agg['Накопленная доля, %'] = dish_agg['Доля выручки, %'].cumsum()

    def abc_class(cumsum):
        if cumsum <= 80: return 'A'
        if cumsum <= 95: return 'B'
        return 'C'

    dish_agg['ABC'] = dish_agg['Накопленная доля, %'].apply(abc_class)
    abc_colors = {'A': PALETTE[1], 'B': PALETTE[2], 'C': PALETTE[3]}

    top_rev = dish_agg.head(TOP_N)
    top_cnt = dish_agg.sort_values('Продаж', ascending=False).head(TOP_N)

    fig, axes = plt.subplots(1, 2, figsize=(18, max(6, TOP_N * 0.45)))
    fig.suptitle(f'🏆 {title_prefix}Топ-{TOP_N} блюд', fontsize=14, fontweight='bold')

    bar_colors_rev = [abc_colors[c] for c in top_rev['ABC']]
    axes[0].barh(top_rev['Блюдо'][::-1], top_rev['Выручка'][::-1], color=bar_colors_rev[::-1])
    for i, (v, cls) in enumerate(zip(top_rev['Выручка'][::-1], top_rev['ABC'][::-1])):
        axes[0].text(v * 0.01, i, f'{rub(v)}  [{cls}]', va='center', fontsize=8.5)
    axes[0].set_title('По выручке (цвет = ABC-класс)')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
    patches = [mpatches.Patch(color=abc_colors[c], label=f'Класс {c}') for c in 'ABC']
    axes[0].legend(handles=patches, frameon=False, fontsize=9)

    bar_colors_cnt = [abc_colors[c] for c in top_cnt['ABC']]
    axes[1].barh(top_cnt['Блюдо'][::-1], top_cnt['Продаж'][::-1], color=bar_colors_cnt[::-1])
    for i, v in enumerate(top_cnt['Продаж'][::-1]):
        axes[1].text(v * 0.01, i, f'{v:,}'.replace(',', ' '), va='center', fontsize=8.5)
    axes[1].set_title('По количеству продаж')

    plt.tight_layout()
    plt.show()

    abc_summary = dish_agg.groupby('ABC').agg(
        Товаров=('Блюдо', 'count'),
        Выручка=('Выручка', 'sum'),
    ).reset_index()
    abc_summary['Доля выручки'] = (abc_summary['Выручка'] / abc_summary['Выручка'].sum() * 100).apply(pct)
    abc_summary['Выручка'] = abc_summary['Выручка'].apply(rub)
    print(f'\n📊 ABC-сводка {title_prefix}:')
    print(abc_summary.to_string(index=False))

    return dish_agg

### 🌐 Все склады

In [ ]:
dish_agg_global = plot_abc(df, title_prefix='[Все склады] ')

### 🏭 По каждому складу

In [ ]:
for wh in warehouses:
    df_wh = df[df['Склад'] == wh]
    if len(df_wh) == 0:
        continue
    print(f'\n📦 Склад: {wh}')
    plot_abc(df_wh, title_prefix=f'[{wh}] ')

## 🎁 8. Программа лояльности <a id='лояльность'></a>

Доля чеков с бонусами, средний чек с/без бонусов, топ-10 клиентов по выручке.

In [ ]:
def plot_loyalty(data, title_prefix=''):
    checks = data.groupby('check_key').agg(
        Сумма=('Сумма со скидкой, р.', 'sum'),
        Скидка=('Скидка, р.', 'sum'),
        Карта=('Номер карты клиента', 'first'),
        Склад=('Склад', 'first'),
        Бонусы=('Тип скидки', lambda x: (x == 'Бонусы').any())
    ).reset_index()

    avg_bonus    = checks[checks['Бонусы']]['Сумма'].mean() if checks['Бонусы'].any() else 0
    avg_no_bonus = checks[~checks['Бонусы']]['Сумма'].mean() if (~checks['Бонусы']).any() else 0

    top_clients = (data[data['Номер карты клиента'].notna()]
                   .assign(**{'Номер карты клиента': lambda d: d['Номер карты клиента'].astype(str)})
                   .groupby('Номер карты клиента')
                   .agg(Выручка=('Сумма со скидкой, р.', 'sum'),
                        Визитов=('check_key', 'nunique'))
                   .reset_index()
                   .sort_values('Выручка', ascending=False)
                   .head(10))

    fig = plt.figure(figsize=(16, 8))
    gs  = GridSpec(2, 2, figure=fig)
    fig.suptitle(f'🎁 {title_prefix}Программа лояльности', fontsize=14, fontweight='bold')

    ax1 = fig.add_subplot(gs[0, 0])
    vals = [int(checks['Бонусы'].sum()), int((~checks['Бонусы']).sum())]
    lbls = [f'С бонусами\n{vals[0]:,}'.replace(',', ' '),
            f'Без бонусов\n{vals[1]:,}'.replace(',', ' ')]
    ax1.pie(vals, labels=lbls, colors=[PALETTE[3], PALETTE[0]],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
    ax1.set_title('Доля чеков с бонусами')

    ax2 = fig.add_subplot(gs[0, 1])
    bars = ax2.bar(['С бонусами', 'Без бонусов'], [avg_bonus, avg_no_bonus],
                   color=[PALETTE[3], PALETTE[0]], width=0.5)
    for bar, val in zip(bars, [avg_bonus, avg_no_bonus]):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                 rub(val), ha='center', fontsize=10, fontweight='bold')
    ax2.set_title('Средний чек')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
    ax2.set_ylim(0, max(avg_bonus, avg_no_bonus) * 1.25 if max(avg_bonus, avg_no_bonus) > 0 else 1)

    ax3 = fig.add_subplot(gs[1, :])
    if len(top_clients) > 0:
        ax3.barh(top_clients['Номер карты клиента'][::-1],
                 top_clients['Выручка'][::-1], color=PALETTE[4])
        for i, (v, vis) in enumerate(zip(top_clients['Выручка'][::-1],
                                          top_clients['Визитов'][::-1])):
            ax3.text(0, i, f'  {rub(v)}  ({vis} визитов)', va='center', fontsize=8.5)
        ax3.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
    else:
        ax3.text(0.5, 0.5, 'Нет данных о картах клиентов',
                 ha='center', va='center', transform=ax3.transAxes, fontsize=12)
    ax3.set_title(f'{title_prefix}Топ-10 клиентов по выручке')

    plt.tight_layout()
    plt.show()

### 🌐 Все склады

In [ ]:
plot_loyalty(df, title_prefix='[Все склады] ')

### 🏭 По каждому складу

In [ ]:
for wh in warehouses:
    df_wh = df[df['Склад'] == wh]
    if len(df_wh) == 0:
        continue
    print(f'\n📦 Склад: {wh}')
    plot_loyalty(df_wh, title_prefix=f'[{wh}] ')

## 🛒 9. Market Basket Analysis — анализ сочетаний товаров <a id='basket'></a>

**Интерпретация лифта:**
- Лифт > 2.0 — очень сильная связь, отличные кандидаты для комбо-предложений
- Лифт 1.5–2.0 — хорошая связь
- Лифт ≈ 1.0 — случайное сочетание

In [ ]:
def plot_basket(data, title_prefix='', min_support=MIN_SUPPORT):
    baskets = data.groupby('check_key')['Блюдо'].apply(set)
    n_baskets = len(baskets)

    # ── Предупреждения о надёжности ──────────────────────────────
    if n_baskets < 500:
        print(f'⛔ [{title_prefix}] Только {n_baskets} чеков — анализ ненадёжен, пропускаем.')
        return None
    if n_baskets < 1300:
        print(f'⚠️ [{title_prefix}] {n_baskets} чеков — результаты ориентировочные, нужно 1300+')

    # ── Адаптивный минимальный порог: не менее 20 совместных покупок ──
    min_count = max(5, int(min_support * n_baskets))

    pair_counts = Counter()
    single_counts = Counter()

    # ── Считаем только чеки с 2+ позициями ───────────────────────
    multi_item_baskets = [items for items in baskets if len(items) >= 2]
    n_for_support = len(baskets)  # знаменатель поддержки — всё равно все чеки

    for items in baskets:
        for item in items:
            single_counts[item] += 1

    for items in multi_item_baskets:
        for pair in combinations(sorted(items), 2):
            pair_counts[pair] += 1

    rows = []
    for (a, b), cnt in pair_counts.items():
        if cnt >= min_count:
            sup_a = single_counts[a] / n_for_support
            sup_b = single_counts[b] / n_for_support
            lift = (cnt / n_for_support) / (sup_a * sup_b) if sup_a * sup_b > 0 else 0
            rows.append({
                'Товар A': a, 'Товар B': b,
                'Совместных покупок': cnt,
                'Поддержка, %': round(cnt / n_for_support * 100, 2),
                'Лифт': round(lift, 2)
            })

    if not rows:
        print(f'   ⚠️ {title_prefix}Нет пар с поддержкой ≥ {min_support*100:.1f}%. Уменьшите MIN_SUPPORT.')
        return None

    pairs_df = pd.DataFrame(rows).sort_values('Лифт', ascending=False).reset_index(drop=True)
    top_pairs = pairs_df.head(20).copy()
    top_pairs['Пара'] = top_pairs['Товар A'] + ' + ' + top_pairs['Товар B']

    fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(top_pairs) * 0.4)))
    fig.suptitle(f'🛒 {title_prefix}Анализ сочетаний товаров (Market Basket)',
                 fontsize=14, fontweight='bold')

    lift_colors = [PALETTE[1] if v >= 2 else (PALETTE[2] if v >= 1.5 else PALETTE[3])
                   for v in top_pairs['Лифт'][::-1]]
    axes[0].barh(top_pairs['Пара'][::-1], top_pairs['Лифт'][::-1], color=lift_colors)
    axes[0].axvline(2.0, color='red', linestyle='--', alpha=0.7, label='Лифт = 2 (сильная связь)')
    axes[0].axvline(1.0, color='gray', linestyle=':', alpha=0.7, label='Лифт = 1 (случайность)')
    for i, v in enumerate(top_pairs['Лифт'][::-1]):
        axes[0].text(v * 0.01, i, f'{v:.2f}', va='center', fontsize=8)
    axes[0].set_title('Топ пар по лифту')
    axes[0].set_xlabel('Лифт')
    axes[0].legend(frameon=False, fontsize=8)

    top_freq = pairs_df.sort_values('Совместных покупок', ascending=False).head(20).copy()
    top_freq['Пара'] = top_freq['Товар A'] + ' + ' + top_freq['Товар B']
    axes[1].barh(top_freq['Пара'][::-1], top_freq['Совместных покупок'][::-1], color=PALETTE[4])
    for i, v in enumerate(top_freq['Совместных покупок'][::-1]):
        axes[1].text(v * 0.01, i, f'{v:,}'.replace(',', ' '), va='center', fontsize=8)
    axes[1].set_title('Топ пар по частоте совместных покупок')

    plt.tight_layout()
    plt.show()

    print(f'\n🔍 {title_prefix}Найдено пар с поддержкой ≥ {min_support*100:.1f}%: {len(pairs_df)}')
    print('Топ-10 пар для комбо-продаж:')
    print(pairs_df[['Товар A','Товар B','Совместных покупок','Поддержка, %','Лифт']].head(10).to_string(index=False))

    return pairs_df

### 🌐 Все склады

In [ ]:
print('=' * 60)
print('🛒 АНАЛИЗ СОЧЕТАНИЙ: ВСЕ СКЛАДЫ')
print('=' * 60)
pairs_df_global = plot_basket(df, title_prefix='[Все склады] ')

### 🏭 По каждому складу

In [ ]:
for wh in warehouses:
    df_wh = df[df['Склад'] == wh]
    if len(df_wh) == 0:
        continue
    print('\n' + '=' * 60)
    print(f'🛒 АНАЛИЗ СОЧЕТАНИЙ: {wh}')
    print('=' * 60)
    plot_basket(df_wh, title_prefix=f'[{wh}] ')

## 🧾 12. Распределение чеков по количеству позиций <a id='check_dist'></a>

Сколько блюд в среднем берут за один визит. Показывает потенциал апселла.

In [ ]:
def plot_check_distribution(data, title_prefix=''):
    # Количество уникальных позиций на чек
    check_sizes = data.groupby('check_key')['Блюдо'].count().reset_index()
    check_sizes.columns = ['check_key', 'Позиций']

    # Группируем: 1, 2, 3, 4, 5+
    check_sizes['Группа'] = check_sizes['Позиций'].apply(
        lambda x: str(x) if x <= 4 else '5+'
    )
    order_groups = ['1', '2', '3', '4', '5+']
    dist = check_sizes['Группа'].value_counts().reindex(order_groups, fill_value=0)
    dist_pct = dist / dist.sum() * 100

    n_checks  = len(check_sizes)
    avg_items = check_sizes['Позиций'].mean()
    median_items = check_sizes['Позиций'].median()
    single_pct = dist_pct.get('1', 0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{title_prefix}Распределение чеков по количеству позиций',
                 fontsize=14, fontweight='bold')

    # Бар: абсолютное количество
    bar_colors = [PALETTE[3] if g == '1' else PALETTE[0] for g in order_groups]
    bars = axes[0].bar(order_groups, dist.values, color=bar_colors, width=0.6, edgecolor='white')
    for bar, v, p in zip(bars, dist.values, dist_pct.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + n_checks*0.005,
                     f'{v:,}\n({p:.1f}%)'.replace(',', ' '),
                     ha='center', fontsize=9, fontweight='bold')
    axes[0].set_xlabel('Количество позиций в чеке')
    axes[0].set_ylabel('Количество чеков')
    axes[0].set_title('Количество чеков')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', ' ')))

    # Кумулятивное распределение
    all_sizes = check_sizes['Позиций'].clip(upper=10)  # ограничим для читаемости
    size_counts = all_sizes.value_counts().sort_index()
    size_pct = size_counts / size_counts.sum() * 100
    cumulative = size_pct.cumsum()
    axes[1].bar(size_counts.index, size_pct.values, color=PALETTE[0], alpha=0.5,
                width=0.7, label='Доля чеков, %')
    ax1b = axes[1].twinx()
    ax1b.plot(cumulative.index, cumulative.values, color=PALETTE[3],
              linewidth=2.5, marker='o', markersize=5, label='Накопленная доля, %')
    ax1b.axhline(80, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax1b.set_ylabel('Накопленная доля, %')
    ax1b.set_ylim(0, 105)
    axes[1].set_xlabel('Позиций в чеке')
    axes[1].set_ylabel('Доля чеков, %')
    axes[1].set_title('Детальное распределение + кумулятивная кривая')
    lines1, labels1 = axes[1].get_legend_handles_labels()
    lines2, labels2 = ax1b.get_legend_handles_labels()
    axes[1].legend(lines1 + lines2, labels1 + labels2, frameon=False, fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f'   Чеков всего:         {n_checks:,}'.replace(',', ' '))
    print(f'   Среднее позиций:     {avg_items:.2f}')
    print(f'   Медиана позиций:     {median_items:.0f}')
    print(f'   Чеков с 1 позицией:  {dist["1"]:,} ({single_pct:.1f}%) — потенциал апселла'.replace(',', ' '))


### 🌐 Все склады

In [ ]:
print('\n📊 Все склады:')
plot_check_distribution(df, title_prefix='[Все склады] ')


### 🏭 По каждому складу

In [ ]:
for wh in warehouses:
    df_wh = df[df['Склад'] == wh]
    if len(df_wh) == 0:
        continue
    print(f'\n📦 Склад: {wh}')
    plot_check_distribution(df_wh, title_prefix=f'[{wh}] ')


## 📊 13. Матрица BCG для блюд <a id='bcg'></a>

**Оси:** доля блюда в выручке (X) и темп роста продаж (Y) — первая vs вторая половина периода.

- 🌟 **Звёзды** — высокая доля, высокий рост → развивать
- 🐄 **Дойные коровы** — высокая доля, низкий рост → беречь маржу
- ❓ **Вопросы** — низкая доля, высокий рост → инвестировать или отказаться
- 🐕 **Собаки** — низкая доля, низкий рост → пересмотреть или убрать

In [ ]:
def plot_bcg(data, title_prefix=''):
    # Делим период пополам
    dates = data['Учетный день'].sort_values()
    mid_date = dates.iloc[len(dates)//2]

    first_half  = data[data['Учетный день'] <  mid_date]
    second_half = data[data['Учетный день'] >= mid_date]

    if len(first_half) == 0 or len(second_half) == 0:
        print(f'   ⚠️ {title_prefix}Недостаточно данных для BCG (нужно минимум 2 периода)')
        return

    rev_first  = first_half.groupby('Блюдо')['Сумма со скидкой, р.'].sum()
    rev_second = second_half.groupby('Блюдо')['Сумма со скидкой, р.'].sum()
    rev_total  = data.groupby('Блюдо')['Сумма со скидкой, р.'].sum()
    cnt_total  = data.groupby('Блюдо')['Блюдо'].count()

    bcg = pd.DataFrame({
        'Выручка': rev_total,
        'Продаж': cnt_total,
        'Выручка_1': rev_first,
        'Выручка_2': rev_second,
    }).fillna(0).reset_index()

    total_rev_all = bcg['Выручка'].sum()
    bcg['Доля, %'] = bcg['Выручка'] / total_rev_all * 100

    # Темп роста: (2-я половина - 1-я) / 1-я * 100
    bcg['Рост, %'] = np.where(
        bcg['Выручка_1'] > 0,
        (bcg['Выручка_2'] - bcg['Выручка_1']) / bcg['Выручка_1'] * 100,
        np.where(bcg['Выручка_2'] > 0, 100.0, 0.0)
    )
    # Ограничиваем выбросы для читаемости графика
    bcg['Рост_clip'] = bcg['Рост, %'].clip(-200, 300)

    median_share  = bcg['Доля, %'].median()
    median_growth = bcg['Рост_clip'].median()

    def bcg_quad(row):
        hi_share  = row['Доля, %']   >= median_share
        hi_growth = row['Рост_clip'] >= median_growth
        if hi_share  and hi_growth:  return '🌟 Звезда'
        if hi_share  and not hi_growth: return '🐄 Дойная корова'
        if not hi_share and hi_growth:  return '❓ Вопрос'
        return '🐕 Собака'

    bcg['Квадрант'] = bcg.apply(bcg_quad, axis=1)

    quad_colors = {
        '🌟 Звезда':        PALETTE[2],
        '🐄 Дойная корова': PALETTE[1],
        '❓ Вопрос':        PALETTE[0],
        '🐕 Собака':        PALETTE[3],
    }

    fig, ax = plt.subplots(figsize=(14, 9))
    fig.suptitle(f'{title_prefix}Матрица BCG — блюда',
                 fontsize=14, fontweight='bold')

    for quad, grp in bcg.groupby('Квадрант'):
        # Размер пузыря = количество продаж
        sizes = (grp['Продаж'] / grp['Продаж'].max() * 600 + 30).clip(lower=30)
        ax.scatter(grp['Доля, %'], grp['Рост_clip'],
                   s=sizes, color=quad_colors[quad], alpha=0.7,
                   edgecolors='white', linewidth=0.8, label=quad, zorder=3)

    # Подписываем топ-15 по выручке
    top_labels = bcg.nlargest(15, 'Выручка')
    for _, row in top_labels.iterrows():
        ax.annotate(row['Блюдо'],
                    xy=(row['Доля, %'], row['Рост_clip']),
                    xytext=(5, 4), textcoords='offset points',
                    fontsize=7.5, alpha=0.9,
                    arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    # Квадранты — разделяющие линии
    ax.axvline(median_share,  color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
    ax.axhline(median_growth, color='gray', linestyle='--', alpha=0.5, linewidth=1.2)
    ax.axhline(0, color='black', linestyle='-', alpha=0.2, linewidth=0.8)

    # Подписи квадрантов
    xlim = ax.get_xlim(); ylim = ax.get_ylim()
    ax.text(xlim[1]*0.98, ylim[1]*0.96, '🌟 Звёзды',        ha='right', va='top',   fontsize=10, color=quad_colors['🌟 Звезда'],        alpha=0.6, fontweight='bold')
    ax.text(xlim[0]+0.01, ylim[1]*0.96, '❓ Вопросы',        ha='left',  va='top',   fontsize=10, color=quad_colors['❓ Вопрос'],        alpha=0.6, fontweight='bold')
    ax.text(xlim[1]*0.98, ylim[0]*0.96, '🐄 Дойные коровы', ha='right', va='bottom', fontsize=10, color=quad_colors['🐄 Дойная корова'], alpha=0.6, fontweight='bold')
    ax.text(xlim[0]+0.01, ylim[0]*0.96, '🐕 Собаки',         ha='left',  va='bottom', fontsize=10, color=quad_colors['🐕 Собака'],        alpha=0.6, fontweight='bold')

    ax.set_xlabel('Доля в выручке, %', fontsize=11)
    ax.set_ylabel('Темп роста выручки, %\n(1-я → 2-я половина периода)', fontsize=11)
    ax.legend(frameon=False, fontsize=10, loc='center right')
    plt.tight_layout()
    plt.show()

    # Сводная таблица по квадрантам
    summary = bcg.groupby('Квадрант').agg(
        Блюд=('Блюдо', 'count'),
        Выручка=('Выручка', 'sum'),
    ).reset_index()
    summary['Доля выручки'] = (summary['Выручка'] / summary['Выручка'].sum() * 100).apply(pct)
    summary['Выручка'] = summary['Выручка'].apply(rub)
    print(f'\n📊 BCG-сводка {title_prefix}:')
    print(summary.to_string(index=False))

    # Топ-5 товаров по каждой ключевой категории
    for quad_name in ['🌟 Звезда', '🐄 Дойная корова', '🐕 Собака']:
        quad_items = bcg[bcg['Квадрант'] == quad_name].nlargest(5, 'Выручка')
        if len(quad_items) == 0:
            continue
        print(f'\n{quad_name} — топ-5 блюд по выручке:')
        for _, row in quad_items.iterrows():
            print(f'  • {row["Блюдо"]:40s}  {rub(row["Выручка"])}  (рост {row["Рост, %"]:+.1f}%)')

    return bcg


### 🌐 Все склады

In [ ]:
print('\n📊 Все склады:')
bcg_global = plot_bcg(df, title_prefix='[Все склады] ')


### 🏭 По каждому складу

In [ ]:
for wh in warehouses:
    df_wh = df[df['Склад'] == wh]
    if len(df_wh) == 0:
        continue
    print(f'\n📦 Склад: {wh}')
    plot_bcg(df_wh, title_prefix=f'[{wh}] ')


## 🌡️ 14. Тепловая карта среднего чека по дням недели и складам <a id='heatmap_avg'></a>

В отличие от тепловой карты выручки — показывает качество визитов, а не трафик.

In [ ]:
day_map2 = {0:'Пн',1:'Вт',2:'Ср',3:'Чт',4:'Пт',5:'Сб',6:'Вс'}
df['День недели'] = df['Учетный день'].dt.dayofweek.map(day_map2)
order_days = ['Пн','Вт','Ср','Чт','Пт','Сб','Вс']

# Считаем средний чек: выручка / количество уникальных чеков
check_rev = df.groupby(['Склад', 'День недели', 'check_key'])['Сумма со скидкой, р.'].sum().reset_index()
avg_check_heatmap = check_rev.groupby(['Склад', 'День недели'])['Сумма со скидкой, р.'].mean().unstack('День недели')
avg_check_heatmap = avg_check_heatmap.reindex(columns=order_days).astype(float)

# Также считаем количество чеков для контекста
n_checks_heatmap = check_rev.groupby(['Склад', 'День недели'])['check_key'].count().unstack('День недели')
n_checks_heatmap = n_checks_heatmap.reindex(columns=order_days).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(20, max(3, len(avg_check_heatmap) * 1.3)))
fig.suptitle('🌡️ Средний чек и трафик по дням недели и складам',
             fontsize=14, fontweight='bold')

# ── Левый: средний чек ────────────────────────────────────────────
im1 = axes[0].imshow(avg_check_heatmap.values, cmap='RdYlGn', aspect='auto')
axes[0].set_xticks(range(len(order_days)))
axes[0].set_xticklabels(order_days)
axes[0].set_yticks(range(len(avg_check_heatmap)))
axes[0].set_yticklabels(avg_check_heatmap.index)
axes[0].set_title('Средний чек, ₽')

vmax1 = np.nanmax(avg_check_heatmap.values)
for i in range(len(avg_check_heatmap)):
    for j in range(len(order_days)):
        val = avg_check_heatmap.iloc[i, j]
        if not np.isnan(val):
            tc = 'white' if val > vmax1 * 0.75 or val < vmax1 * 0.3 else 'black'
            axes[0].text(j, i, fmt_rub(val), ha='center', va='center',
                         fontsize=9, color=tc, fontweight='bold')

plt.colorbar(im1, ax=axes[0], label='Средний чек, ₽', shrink=0.8)

# ── Правый: количество чеков (трафик) ─────────────────────────────
im2 = axes[1].imshow(n_checks_heatmap.values, cmap='Blues', aspect='auto')
axes[1].set_xticks(range(len(order_days)))
axes[1].set_xticklabels(order_days)
axes[1].set_yticks(range(len(n_checks_heatmap)))
axes[1].set_yticklabels(n_checks_heatmap.index)
axes[1].set_title('Количество чеков (трафик)')

vmax2 = np.nanmax(n_checks_heatmap.values)
for i in range(len(n_checks_heatmap)):
    for j in range(len(order_days)):
        val = n_checks_heatmap.iloc[i, j]
        if not np.isnan(val):
            tc = 'white' if val > vmax2 * 0.6 else 'black'
            axes[1].text(j, i, f'{int(val):,}'.replace(',', ' '),
                         ha='center', va='center', fontsize=9, color=tc, fontweight='bold')

plt.colorbar(im2, ax=axes[1], label='Чеков', shrink=0.8)

plt.tight_layout()
plt.show()

# ── Таблица: топ-3 дня по среднему чеку для каждого склада ────────
print('\n📊 Лучшие дни по среднему чеку (топ-3 для каждого склада):')
for wh in avg_check_heatmap.index:
    row = avg_check_heatmap.loc[wh].dropna().sort_values(ascending=False)
    top3 = ', '.join([f'{d}: {rub(v)}' for d, v in row.head(3).items()])
    print(f'  {wh}: {top3}')


## 🔍 15. Уникальность ассортимента по складам <a id='unique_dishes'></a>

Какие блюда продаются только на одном складе — эксклюзив. Какие — на всех. Где пересечения максимальны.

In [ ]:
# ── Множества блюд по складам ────────────────────────────────────
dish_sets = df.groupby('Склад')['Блюдо'].apply(set)
all_dishes = set(df['Блюдо'].unique())

# Для каждого блюда — на скольких складах оно продаётся
dish_wh_count = df.groupby('Блюдо')['Склад'].nunique().reset_index()
dish_wh_count.columns = ['Блюдо', 'Складов']
dish_wh_count = dish_wh_count.merge(
    df.groupby('Блюдо')['Сумма со скидкой, р.'].sum().reset_index(),
    on='Блюдо'
).merge(
    df.groupby('Блюдо')['Блюдо'].count().reset_index(name='Продаж'),
    on='Блюдо'
)

n_warehouses = len(warehouses)

# ── График 1: распределение блюд по охвату складов ────────────────
coverage_dist = dish_wh_count['Складов'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('🔍 Уникальность ассортимента по складам', fontsize=14, fontweight='bold')

cov_colors = [PALETTE[3] if x == 1 else (PALETTE[1] if x == n_warehouses else PALETTE[0])
              for x in coverage_dist.index]
bars = axes[0].bar([str(x) for x in coverage_dist.index], coverage_dist.values,
                   color=cov_colors, width=0.6, edgecolor='white')
for bar, v in zip(bars, coverage_dist.values):
    pct_val = v / len(dish_wh_count) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{v}\n({pct_val:.0f}%)', ha='center', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Количество складов, где продаётся блюдо')
axes[0].set_ylabel('Количество блюд')
axes[0].set_title('Сколько складов продают каждое блюдо')

# Добавляем метки на оси X
x_labels = []
for x in coverage_dist.index:
    if x == 1: x_labels.append(f'{x}\n(эксклюзив)')
    elif x == n_warehouses: x_labels.append(f'{x}\n(везде)')
    else: x_labels.append(str(x))
axes[0].set_xticklabels(x_labels)

# ── График 2: уникальные блюда по складам (эксклюзив) ─────────────
exclusive = dish_wh_count[dish_wh_count['Складов'] == 1].copy()
exclusive = exclusive.merge(
    df[['Блюдо','Склад']].drop_duplicates(), on='Блюдо'
)
excl_by_wh = exclusive.groupby('Склад').size().reset_index(name='Эксклюзивных блюд')
excl_by_wh = excl_by_wh.sort_values('Эксклюзивных блюд', ascending=True)

wh_colors = [PALETTE[i % len(PALETTE)] for i in range(len(excl_by_wh))]
axes[1].barh(excl_by_wh['Склад'], excl_by_wh['Эксклюзивных блюд'], color=wh_colors)
for i, v in enumerate(excl_by_wh['Эксклюзивных блюд']):
    axes[1].text(v + 0.2, i, str(v), va='center', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Количество эксклюзивных блюд')
axes[1].set_title('Эксклюзивные блюда по складам')

plt.tight_layout()
plt.show()

# ── Топ эксклюзивных блюд по выручке для каждого склада ───────────
print('\n🏅 Топ-5 эксклюзивных блюд по выручке для каждого склада:')
for wh in warehouses:
    wh_excl = exclusive[exclusive['Склад'] == wh].sort_values(
        'Сумма со скидкой, р.', ascending=False
    ).head(5)
    if len(wh_excl) == 0:
        print(f'\n  📦 {wh}: нет эксклюзивных блюд')
        continue
    print(f'\n  📦 {wh} ({len(exclusive[exclusive["Склад"]==wh])} эксклюзивных):')
    for _, r in wh_excl.iterrows():
        print(f'     {r["Блюдо"]:<40} {rub(r["Сумма со скидкой, р."]):>14}  ({int(r["Продаж"])} продаж)')

# ── Матрица пересечений ───────────────────────────────────────────
if n_warehouses > 1:
    print('\n🔄 Матрица пересечений ассортимента (% общих блюд):')
    overlap_data = []
    for wh_a in warehouses:
        row_data = []
        set_a = dish_sets.get(wh_a, set())
        for wh_b in warehouses:
            set_b = dish_sets.get(wh_b, set())
            if wh_a == wh_b:
                row_data.append(100.0)
            else:
                overlap = len(set_a & set_b) / len(set_a | set_b) * 100 if set_a | set_b else 0
                row_data.append(round(overlap, 1))
        overlap_data.append(row_data)

    overlap_df = pd.DataFrame(overlap_data, index=warehouses, columns=warehouses)

    fig, ax = plt.subplots(figsize=(max(6, n_warehouses * 1.5), max(5, n_warehouses * 1.3)))
    fig.suptitle('🔄 Матрица пересечений ассортимента (% общих блюд, Jaccard)',
                 fontsize=13, fontweight='bold')
    im = ax.imshow(overlap_df.values, cmap='YlOrRd', vmin=0, vmax=100, aspect='auto')
    ax.set_xticks(range(n_warehouses))
    ax.set_xticklabels(warehouses, rotation=30, ha='right')
    ax.set_yticks(range(n_warehouses))
    ax.set_yticklabels(warehouses)
    for i in range(n_warehouses):
        for j in range(n_warehouses):
            val = overlap_df.iloc[i, j]
            tc = 'white' if val > 60 else 'black'
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                    fontsize=11, fontweight='bold', color=tc)
    plt.colorbar(im, ax=ax, label='% общих блюд (Jaccard)', shrink=0.8)
    plt.tight_layout()
    plt.show()

    print('\n' + overlap_df.to_string())

# ── Блюда на ВСЕХ складах ─────────────────────────────────────────
universal = dish_wh_count[dish_wh_count['Складов'] == n_warehouses].sort_values(
    'Сумма со скидкой, р.', ascending=False
)
print(f'\n🌐 Блюд, которые продаются на ВСЕХ {n_warehouses} складах: {len(universal)}')
if len(universal) > 0:
    print('   Топ-10 по выручке:')
    for _, r in universal.head(10).iterrows():
        print(f'   {r["Блюдо"]:<40} {rub(r["Сумма со скидкой, р."]):>14}  ({int(r["Продаж"])} продаж)')


## 🗓 10. Тепловая карта — выручка по дням недели <a id='хeatmap'></a>

In [ ]:
day_map = {0:'Пн',1:'Вт',2:'Ср',3:'Чт',4:'Пт',5:'Сб',6:'Вс'}
df['День недели'] = df['Учетный день'].dt.dayofweek.map(day_map)
order = ['Пн','Вт','Ср','Чт','Пт','Сб','Вс']

heatmap_data = df.groupby(['Склад', 'День недели'])['Сумма со скидкой, р.'].sum().unstack('День недели')
heatmap_data = heatmap_data.reindex(columns=order).astype(float)

fig, ax = plt.subplots(figsize=(12, max(3, len(heatmap_data) * 1.2)))
fig.suptitle('🗓 Выручка по дням недели и складам', fontsize=14, fontweight='bold')

im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order)
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index)

for i in range(len(heatmap_data)):
    for j in range(len(order)):
        val = heatmap_data.iloc[i, j]
        if not np.isnan(val):
            text_color = 'white' if val > heatmap_data.values[~np.isnan(heatmap_data.values)].max() * 0.6 else 'black'
            ax.text(j, i, fmt_rub(val), ha='center', va='center',
                    fontsize=9, color=text_color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Выручка, ₽', shrink=0.8)
plt.tight_layout()
plt.show()

## 👤 15. Анализ кассиров <a id='cashiers'></a>

Рейтинг кассиров по выручке, среднему чеку, марже и вовлечённости в программу лояльности.


In [ ]:
if not has_cashier:
    print('⚠️ Колонка «Кассир» не найдена в данных — раздел пропущен.')
else:
    cashier_agg = df.groupby(cashier_col).agg(
        Выручка=('Сумма со скидкой, р.', 'sum'),
        Чеков=('check_key', 'nunique'),
        **({'Маржа': ('Маржа, р.', 'sum')} if has_cost else {})
    ).reset_index()
    cashier_agg['Средний чек'] = cashier_agg['Выручка'] / cashier_agg['Чеков']
    if has_cost:
        cashier_agg['Маржа, %'] = cashier_agg['Маржа'] / cashier_agg['Выручка'] * 100

    # Медианный средний чек по всем кассирам — для upsell-индекса
    median_avg_check = cashier_agg['Средний чек'].median()
    cashier_agg['Upsell-индекс'] = cashier_agg['Средний чек'] / median_avg_check

    # Доля чеков с бонусами по кассиру
    bonus_by_cashier = (
        df[df['Тип скидки'] == 'Бонусы']
        .groupby(cashier_col)['check_key']
        .nunique()
        .reset_index(name='Бонусных чеков')
    )
    cashier_agg = cashier_agg.merge(bonus_by_cashier, on=cashier_col, how='left')
    cashier_agg['Бонусных чеков'] = cashier_agg['Бонусных чеков'].fillna(0).astype(int)
    cashier_agg['Доля бонусов, %'] = cashier_agg['Бонусных чеков'] / cashier_agg['Чеков'] * 100
    cashier_agg = cashier_agg.sort_values('Выручка', ascending=False).reset_index(drop=True)

    top_n_cashiers = min(20, len(cashier_agg))
    top_c = cashier_agg.head(top_n_cashiers)

    ncols = 3 if has_cost else 2
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, max(5, top_n_cashiers * 0.45)))
    fig.suptitle(f'👤 Рейтинг кассиров (топ-{top_n_cashiers})', fontsize=14, fontweight='bold')
    colors = [PALETTE[i % len(PALETTE)] for i in range(top_n_cashiers)]

    axes[0].barh(top_c[cashier_col][::-1], top_c['Выручка'][::-1], color=colors[::-1])
    for i, v in enumerate(top_c['Выручка'][::-1]):
        axes[0].text(v * 0.01, i, rub(v), va='center', fontsize=8)
    axes[0].set_title('Выручка')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))

    axes[1].barh(top_c[cashier_col][::-1], top_c['Средний чек'][::-1], color=colors[::-1])
    for i, (v, idx) in enumerate(zip(top_c['Средний чек'][::-1], top_c['Upsell-индекс'][::-1])):
        axes[1].text(v * 0.01, i, f'{rub(v)}  [×{idx:.2f}]', va='center', fontsize=8)
    axes[1].set_title('Средний чек [upsell-индекс]')
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))

    if has_cost:
        bar_colors_m = [PALETTE[1] if v >= 40 else PALETTE[3] for v in top_c['Маржа, %'][::-1]]
        axes[2].barh(top_c[cashier_col][::-1], top_c['Маржа, %'][::-1], color=bar_colors_m)
        for i, v in enumerate(top_c['Маржа, %'][::-1]):
            axes[2].text(v * 0.01, i, pct(v), va='center', fontsize=8)
        axes[2].set_title('Маржинальность')
        axes[2].axvline(40, color='gray', linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

    # ── График: доля бонусных чеков по кассиру ──────────────────────
    fig2, ax2 = plt.subplots(figsize=(10, max(4, top_n_cashiers * 0.45)))
    fig2.suptitle('🎁 Вовлечённость кассиров в программу лояльности', fontsize=13, fontweight='bold')
    bar_cols2 = [PALETTE[1] if v >= cashier_agg['Доля бонусов, %'].median() else PALETTE[3]
                 for v in top_c['Доля бонусов, %'][::-1]]
    ax2.barh(top_c[cashier_col][::-1], top_c['Доля бонусов, %'][::-1], color=bar_cols2)
    for i, (v, b) in enumerate(zip(top_c['Доля бонусов, %'][::-1], top_c['Бонусных чеков'][::-1])):
        ax2.text(v + 0.3, i, f'{v:.1f}%  ({b} чеков)', va='center', fontsize=8.5)
    ax2.axvline(cashier_agg['Доля бонусов, %'].median(), color='gray', linestyle='--',
                alpha=0.7, label=f'Медиана: {cashier_agg["Доля бонусов, %"].median():.1f}%')
    ax2.set_xlabel('Доля чеков с бонусами, %')
    ax2.legend(frameon=False)
    plt.tight_layout()
    plt.show()

    print('\n📊 Сводка по кассирам:')
    display_cols = [cashier_col, 'Выручка', 'Чеков', 'Средний чек', 'Upsell-индекс', 'Доля бонусов, %']
    if has_cost: display_cols += ['Маржа, %']
    tbl = cashier_agg[display_cols].copy()
    tbl['Выручка'] = tbl['Выручка'].apply(rub)
    tbl['Средний чек'] = tbl['Средний чек'].apply(rub)
    tbl['Upsell-индекс'] = tbl['Upsell-индекс'].apply(lambda x: f'{x:.2f}')
    tbl['Доля бонусов, %'] = tbl['Доля бонусов, %'].apply(pct)
    if has_cost: tbl['Маржа, %'] = tbl['Маржа, %'].apply(pct)
    print(tbl.to_string(index=False))


### 🏭 Анализ кассиров — по каждому складу

In [ ]:
# ── Анализ кассиров по каждому складу ──────────────────────────────
if not has_cashier:
    print('⚠️ Колонка «Кассир» не найдена — разбивка по складам пропущена.')
else:
    for wh in warehouses:
        df_wh = df[df['Склад'] == wh]
        if len(df_wh) == 0:
            continue
        print(f'\n{"="*60}')
        print(f'👤 Кассиры — склад: {wh}')
        print(f'{"="*60}')

        wh_cashier = df_wh.groupby(cashier_col).agg(
            Выручка=('Сумма со скидкой, р.', 'sum'),
            Чеков=('check_key', 'nunique'),
            **({'Маржа': ('Маржа, р.', 'sum')} if has_cost else {})
        ).reset_index()
        wh_cashier['Средний чек'] = wh_cashier['Выручка'] / wh_cashier['Чеков']
        if has_cost:
            wh_cashier['Маржа, %'] = wh_cashier['Маржа'] / wh_cashier['Выручка'] * 100

        median_avg_check_wh = wh_cashier['Средний чек'].median()
        wh_cashier['Upsell-индекс'] = wh_cashier['Средний чек'] / median_avg_check_wh

        bonus_by_cashier_wh = (
            df_wh[df_wh['Тип скидки'] == 'Бонусы']
            .groupby(cashier_col)['check_key']
            .nunique()
            .reset_index(name='Бонусных чеков')
        )
        wh_cashier = wh_cashier.merge(bonus_by_cashier_wh, on=cashier_col, how='left')
        wh_cashier['Бонусных чеков'] = wh_cashier['Бонусных чеков'].fillna(0).astype(int)
        wh_cashier['Доля бонусов, %'] = wh_cashier['Бонусных чеков'] / wh_cashier['Чеков'] * 100
        wh_cashier = wh_cashier.sort_values('Выручка', ascending=False).reset_index(drop=True)

        top_n_c = min(20, len(wh_cashier))
        top_c_wh = wh_cashier.head(top_n_c)

        ncols = 3 if has_cost else 2
        fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, max(5, top_n_c * 0.45)))
        fig.suptitle(f'👤 [{wh}] Рейтинг кассиров (топ-{top_n_c})', fontsize=14, fontweight='bold')
        colors = [PALETTE[i % len(PALETTE)] for i in range(top_n_c)]

        axes[0].barh(top_c_wh[cashier_col][::-1], top_c_wh['Выручка'][::-1], color=colors[::-1])
        for i, v in enumerate(top_c_wh['Выручка'][::-1]):
            axes[0].text(v * 0.01, i, rub(v), va='center', fontsize=8)
        axes[0].set_title('Выручка')
        axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))

        axes[1].barh(top_c_wh[cashier_col][::-1], top_c_wh['Средний чек'][::-1], color=colors[::-1])
        for i, (v, idx) in enumerate(zip(top_c_wh['Средний чек'][::-1], top_c_wh['Upsell-индекс'][::-1])):
            axes[1].text(v * 0.01, i, f'{rub(v)}  [×{idx:.2f}]', va='center', fontsize=8)
        axes[1].set_title('Средний чек [upsell-индекс]')
        axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))

        if has_cost:
            bar_colors_m = [PALETTE[1] if v >= 40 else PALETTE[3] for v in top_c_wh['Маржа, %'][::-1]]
            axes[2].barh(top_c_wh[cashier_col][::-1], top_c_wh['Маржа, %'][::-1], color=bar_colors_m)
            for i, v in enumerate(top_c_wh['Маржа, %'][::-1]):
                axes[2].text(v * 0.01, i, pct(v), va='center', fontsize=8)
            axes[2].set_title('Маржинальность')
            axes[2].axvline(40, color='gray', linestyle='--', alpha=0.6)

        plt.tight_layout()
        plt.show()

        # График: доля бонусных чеков
        fig2, ax2 = plt.subplots(figsize=(10, max(4, top_n_c * 0.45)))
        fig2.suptitle(f'🎁 [{wh}] Вовлечённость кассиров в программу лояльности',
                      fontsize=13, fontweight='bold')
        bar_cols2 = [PALETTE[1] if v >= wh_cashier['Доля бонусов, %'].median() else PALETTE[3]
                     for v in top_c_wh['Доля бонусов, %'][::-1]]
        ax2.barh(top_c_wh[cashier_col][::-1], top_c_wh['Доля бонусов, %'][::-1], color=bar_cols2)
        for i, (v, b) in enumerate(zip(top_c_wh['Доля бонусов, %'][::-1], top_c_wh['Бонусных чеков'][::-1])):
            ax2.text(v + 0.3, i, f'{v:.1f}%  ({b} чеков)', va='center', fontsize=8.5)
        ax2.axvline(wh_cashier['Доля бонусов, %'].median(), color='gray', linestyle='--',
                    alpha=0.7, label=f'Медиана: {wh_cashier["Доля бонусов, %"].median():.1f}%')
        ax2.set_xlabel('Доля чеков с бонусами, %')
        ax2.legend(frameon=False)
        plt.tight_layout()
        plt.show()

        print(f'\n📊 Сводка по кассирам — {wh}:')
        display_cols = [cashier_col, 'Выручка', 'Чеков', 'Средний чек', 'Upsell-индекс', 'Доля бонусов, %']
        if has_cost: display_cols += ['Маржа, %']
        tbl = wh_cashier[display_cols].copy()
        tbl['Выручка'] = tbl['Выручка'].apply(rub)
        tbl['Средний чек'] = tbl['Средний чек'].apply(rub)
        tbl['Upsell-индекс'] = tbl['Upsell-индекс'].apply(lambda x: f'{x:.2f}')
        tbl['Доля бонусов, %'] = tbl['Доля бонусов, %'].apply(pct)
        if has_cost: tbl['Маржа, %'] = tbl['Маржа, %'].apply(pct)
        print(tbl.to_string(index=False))


## 🕐 16. Почасовой анализ трафика <a id='hourly'></a>

Распределение чеков, выручки и маржи по часам дня. Тепловая карта «час × день недели».


In [ ]:
if not has_time:
    print('⚠️ Колонка «Время открытия» не найдена в данных — раздел пропущен.')
else:
    # ── Распределение чеков и выручки по часам ──────────────────────
    hourly = df.groupby('Час открытия').agg(
        Выручка=('Сумма со скидкой, р.', 'sum'),
        Чеков=('check_key', 'nunique'),
        **({'Маржа': ('Маржа, р.', 'sum')} if has_cost else {})
    ).reset_index()
    hourly = hourly.sort_values('Час открытия')
    if has_cost:
        hourly['Маржа, %'] = hourly['Маржа'] / hourly['Выручка'] * 100

    ncols = 3 if has_cost else 2
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 5))
    fig.suptitle('🕐 Почасовой анализ — все склады', fontsize=14, fontweight='bold')

    axes[0].bar(hourly['Час открытия'], hourly['Чеков'], color=PALETTE[0], width=0.7)
    axes[0].set_title('Количество чеков по часам')
    axes[0].set_xlabel('Час дня')
    axes[0].set_ylabel('Чеков')
    axes[0].set_xticks(hourly['Час открытия'])

    axes[1].bar(hourly['Час открытия'], hourly['Выручка'], color=PALETTE[1], width=0.7)
    axes[1].set_title('Выручка по часам')
    axes[1].set_xlabel('Час дня')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
    axes[1].set_xticks(hourly['Час открытия'])

    if has_cost:
        bar_cols_h = [PALETTE[1] if v >= 40 else PALETTE[3] for v in hourly['Маржа, %']]
        axes[2].bar(hourly['Час открытия'], hourly['Маржа, %'], color=bar_cols_h, width=0.7)
        axes[2].axhline(40, color='gray', linestyle='--', alpha=0.6, label='Порог 40%')
        axes[2].set_title('Маржинальность по часам')
        axes[2].set_xlabel('Час дня')
        axes[2].set_ylabel('%')
        axes[2].set_xticks(hourly['Час открытия'])
        axes[2].legend(frameon=False)

    plt.tight_layout()
    plt.show()

    # ── Тепловая карта: час × день недели ───────────────────────────
    day_order = ['Пн','Вт','Ср','Чт','Пт','Сб','Вс']
    day_map_h = {0:'Пн',1:'Вт',2:'Ср',3:'Чт',4:'Пт',5:'Сб',6:'Вс'}
    df['_dow'] = df['Учетный день'].dt.dayofweek.map(day_map_h)

    hm_checks = (
        df.groupby(['Час открытия', '_dow'])['check_key']
        .nunique()
        .unstack('_dow')
        .reindex(columns=day_order)
        .astype(float)
    )
    hm_rev = (
        df.groupby(['Час открытия', '_dow'])['Сумма со скидкой, р.']
        .sum()
        .unstack('_dow')
        .reindex(columns=day_order)
        .astype(float)
    )

    fig2, axes2 = plt.subplots(1, 2, figsize=(18, max(6, len(hm_checks) * 0.5)))
    fig2.suptitle('🗓 Тепловая карта: час × день недели', fontsize=14, fontweight='bold')

    for ax_, data_, title_, fmt_fn in [
        (axes2[0], hm_checks, 'Количество чеков', lambda v: f'{int(v):,}'.replace(',', ' ')),
        (axes2[1], hm_rev,    'Выручка',          lambda v: fmt_rub(v)),
    ]:
        im_ = ax_.imshow(data_.values, cmap='YlOrRd', aspect='auto')
        ax_.set_xticks(range(len(day_order)))
        ax_.set_xticklabels(day_order)
        ax_.set_yticks(range(len(data_)))
        ax_.set_yticklabels([f'{h}:00' for h in data_.index])
        ax_.set_title(title_)
        vmax_ = np.nanmax(data_.values) if not np.all(np.isnan(data_.values)) else 1
        for i in range(len(data_)):
            for j in range(len(day_order)):
                val = data_.iloc[i, j]
                if not np.isnan(val):
                    tc = 'white' if val > vmax_ * 0.6 else 'black'
                    ax_.text(j, i, fmt_fn(val), ha='center', va='center', fontsize=8, color=tc)
        plt.colorbar(im_, ax=ax_, shrink=0.7)

    plt.tight_layout()
    plt.show()

    # ── Пиковые часы ────────────────────────────────────────────────
    peak_hour_rev  = hourly.loc[hourly['Выручка'].idxmax(), 'Час открытия']
    peak_hour_cnt  = hourly.loc[hourly['Чеков'].idxmax(),   'Час открытия']
    print(f'\n⏰ Пик по выручке:  {peak_hour_rev}:00–{peak_hour_rev+1}:00')
    print(f'⏰ Пик по трафику:  {peak_hour_cnt}:00–{peak_hour_cnt+1}:00')
    if has_cost:
        peak_hour_margin = hourly.loc[hourly['Маржа, %'].idxmax(), 'Час открытия']
        print(f'⏰ Пик по марже:    {peak_hour_margin}:00–{peak_hour_margin+1}:00')


### 🏭 Почасовой анализ — по каждому складу

In [ ]:
# ── Почасовой анализ по каждому складу ─────────────────────────────
if not has_time:
    print('⚠️ Колонка «Время открытия» не найдена — разбивка по складам пропущена.')
else:
    day_order_wh = ['Пн','Вт','Ср','Чт','Пт','Сб','Вс']
    day_map_wh = {0:'Пн',1:'Вт',2:'Ср',3:'Чт',4:'Пт',5:'Сб',6:'Вс'}

    for wh in warehouses:
        df_wh = df[df['Склад'] == wh]
        if len(df_wh) == 0:
            continue
        print(f'\n{"="*60}')
        print(f'🕐 Почасовой анализ — склад: {wh}')
        print(f'{"="*60}')

        hourly_wh = df_wh.groupby('Час открытия').agg(
            Выручка=('Сумма со скидкой, р.', 'sum'),
            Чеков=('check_key', 'nunique'),
            **({'Маржа': ('Маржа, р.', 'sum')} if has_cost else {})
        ).reset_index()
        hourly_wh = hourly_wh.sort_values('Час открытия')
        if has_cost:
            hourly_wh['Маржа, %'] = hourly_wh['Маржа'] / hourly_wh['Выручка'] * 100

        ncols = 3 if has_cost else 2
        fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 5))
        fig.suptitle(f'🕐 [{wh}] Почасовой анализ', fontsize=14, fontweight='bold')

        axes[0].bar(hourly_wh['Час открытия'], hourly_wh['Чеков'], color=PALETTE[0], width=0.7)
        axes[0].set_title('Количество чеков по часам')
        axes[0].set_xlabel('Час дня')
        axes[0].set_ylabel('Чеков')
        axes[0].set_xticks(hourly_wh['Час открытия'])

        axes[1].bar(hourly_wh['Час открытия'], hourly_wh['Выручка'], color=PALETTE[1], width=0.7)
        axes[1].set_title('Выручка по часам')
        axes[1].set_xlabel('Час дня')
        axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
        axes[1].set_xticks(hourly_wh['Час открытия'])

        if has_cost:
            bar_cols_h = [PALETTE[1] if v >= 40 else PALETTE[3] for v in hourly_wh['Маржа, %']]
            axes[2].bar(hourly_wh['Час открытия'], hourly_wh['Маржа, %'], color=bar_cols_h, width=0.7)
            axes[2].axhline(40, color='gray', linestyle='--', alpha=0.6, label='Порог 40%')
            axes[2].set_title('Маржинальность по часам')
            axes[2].set_xlabel('Час дня')
            axes[2].set_ylabel('%')
            axes[2].set_xticks(hourly_wh['Час открытия'])
            axes[2].legend(frameon=False)

        plt.tight_layout()
        plt.show()

        # Тепловая карта: час × день недели
        df_wh['_dow_wh'] = df_wh['Учетный день'].dt.dayofweek.map(day_map_wh)

        hm_checks_wh = (
            df_wh.groupby(['Час открытия', '_dow_wh'])['check_key']
            .nunique()
            .unstack('_dow_wh')
            .reindex(columns=day_order_wh)
            .astype(float)
        )
        hm_rev_wh = (
            df_wh.groupby(['Час открытия', '_dow_wh'])['Сумма со скидкой, р.']
            .sum()
            .unstack('_dow_wh')
            .reindex(columns=day_order_wh)
            .astype(float)
        )

        fig2, axes2 = plt.subplots(1, 2, figsize=(18, max(6, len(hm_checks_wh) * 0.5)))
        fig2.suptitle(f'🗓 [{wh}] Тепловая карта: час × день недели', fontsize=14, fontweight='bold')

        for ax_, data_, title_, fmt_fn in [
            (axes2[0], hm_checks_wh, 'Количество чеков', lambda v: f'{int(v):,}'.replace(',', ' ')),
            (axes2[1], hm_rev_wh,    'Выручка',          lambda v: fmt_rub(v)),
        ]:
            im_ = ax_.imshow(data_.values, cmap='YlOrRd', aspect='auto')
            ax_.set_xticks(range(len(day_order_wh)))
            ax_.set_xticklabels(day_order_wh)
            ax_.set_yticks(range(len(data_)))
            ax_.set_yticklabels([f'{h}:00' for h in data_.index])
            ax_.set_title(title_)
            vmax_ = np.nanmax(data_.values) if not np.all(np.isnan(data_.values)) else 1
            for i in range(len(data_)):
                for j in range(len(day_order_wh)):
                    val = data_.iloc[i, j]
                    if not np.isnan(val):
                        tc = 'white' if val > vmax_ * 0.6 else 'black'
                        ax_.text(j, i, fmt_fn(val), ha='center', va='center', fontsize=8, color=tc)
            plt.colorbar(im_, ax=ax_, shrink=0.7)

        plt.tight_layout()
        plt.show()

        # Пиковые часы
        if len(hourly_wh) > 0:
            peak_rev_wh  = hourly_wh.loc[hourly_wh['Выручка'].idxmax(), 'Час открытия']
            peak_cnt_wh  = hourly_wh.loc[hourly_wh['Чеков'].idxmax(),   'Час открытия']
            print(f'⏰ [{wh}] Пик по выручке:  {peak_rev_wh}:00–{peak_rev_wh+1}:00')
            print(f'⏰ [{wh}] Пик по трафику:  {peak_cnt_wh}:00–{peak_cnt_wh+1}:00')
            if has_cost and len(hourly_wh) > 0:
                peak_margin_wh = hourly_wh.loc[hourly_wh['Маржа, %'].idxmax(), 'Час открытия']
                print(f'⏰ [{wh}] Пик по марже:    {peak_margin_wh}:00–{peak_margin_wh+1}:00')


## 💹 17. ABC-анализ по марже <a id='abc_margin'></a>

Ранжирование блюд по **сумме маржи** — дополняет ABC по выручке. Высоковыручное блюдо с низкой маржой может оказаться ниже в этом рейтинге.


In [ ]:
if not has_cost:
    print('⚠️ Себестоимость отсутствует — ABC по марже недоступен.')
else:
    margin_agg = df.groupby('Блюдо').agg(
        Выручка=('Сумма со скидкой, р.', 'sum'),
        Маржа=('Маржа, р.', 'sum'),
        Продаж=('Блюдо', 'count'),
    ).reset_index()
    margin_agg = margin_agg.sort_values('Маржа', ascending=False).reset_index(drop=True)
    margin_agg['Маржа, %'] = margin_agg['Маржа'] / margin_agg['Выручка'] * 100
    margin_agg['Доля маржи, %'] = margin_agg['Маржа'] / margin_agg['Маржа'].sum() * 100
    margin_agg['Накопл. доля маржи, %'] = margin_agg['Доля маржи, %'].cumsum()

    def abc_m(cumsum):
        if cumsum <= 80: return 'A'
        if cumsum <= 95: return 'B'
        return 'C'
    margin_agg['ABC_маржа'] = margin_agg['Накопл. доля маржи, %'].apply(abc_m)
    abc_colors = {'A': PALETTE[1], 'B': PALETTE[2], 'C': PALETTE[3]}

    top_m = margin_agg.head(TOP_N)

    fig, axes = plt.subplots(1, 2, figsize=(18, max(6, TOP_N * 0.45)))
    fig.suptitle(f'💹 ABC-анализ по марже — топ-{TOP_N} блюд', fontsize=14, fontweight='bold')

    bar_colors_m = [abc_colors[c] for c in top_m['ABC_маржа']]
    axes[0].barh(top_m['Блюдо'][::-1], top_m['Маржа'][::-1], color=bar_colors_m[::-1])
    for i, (v, cls) in enumerate(zip(top_m['Маржа'][::-1], top_m['ABC_маржа'][::-1])):
        axes[0].text(v * 0.01, i, f'{rub(v)}  [{cls}]', va='center', fontsize=8.5)
    axes[0].set_title('По сумме маржи (цвет = ABC-класс)')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
    patches = [mpatches.Patch(color=abc_colors[c], label=f'Класс {c}') for c in 'ABC']
    axes[0].legend(handles=patches, frameon=False, fontsize=9)

    axes[1].barh(top_m['Блюдо'][::-1], top_m['Маржа, %'][::-1],
                 color=[PALETTE[1] if v >= 40 else PALETTE[3] for v in top_m['Маржа, %'][::-1]])
    for i, v in enumerate(top_m['Маржа, %'][::-1]):
        axes[1].text(v * 0.01, i, pct(v), va='center', fontsize=8.5)
    axes[1].axvline(40, color='gray', linestyle='--', alpha=0.6, label='Порог 40%')
    axes[1].set_title('Маржинальность, %')
    axes[1].legend(frameon=False, fontsize=9)

    plt.tight_layout()
    plt.show()

    # ── Сравнение ABC по выручке vs по марже ────────────────────────
    if 'dish_agg_global' in dir() and dish_agg_global is not None and len(dish_agg_global) > 0:
        compare = margin_agg[['Блюдо','ABC_маржа']].merge(
            dish_agg_global[['Блюдо','ABC']].rename(columns={'ABC':'ABC_выручка'}),
            on='Блюдо', how='inner'
        )
        diff = compare[compare['ABC_маржа'] != compare['ABC_выручка']]
        if len(diff) > 0:
            print(f'\n🔄 Блюда, у которых ABC-класс по марже отличается от класса по выручке ({len(diff)} шт.):')
            print(f'{"Блюдо":<42} {"ABC выручка":>12} {"ABC маржа":>10}')
            print('-' * 66)
            for _, row in diff.sort_values('ABC_маржа').iterrows():
                print(f'{row["Блюдо"]:<42} {row["ABC_выручка"]:>12} {row["ABC_маржа"]:>10}')
        else:
            print('\n✅ ABC-классы по марже и выручке полностью совпадают.')

    abc_m_summary = margin_agg.groupby('ABC_маржа').agg(
        Блюд=('Блюдо','count'),
        Маржа=('Маржа','sum'),
    ).reset_index()
    abc_m_summary['Доля маржи'] = (abc_m_summary['Маржа'] / abc_m_summary['Маржа'].sum() * 100).apply(pct)
    abc_m_summary['Маржа'] = abc_m_summary['Маржа'].apply(rub)
    print('\n📊 ABC-сводка по марже:')
    print(abc_m_summary.to_string(index=False))


### 🏭 ABC-анализ по марже — по каждому складу

In [ ]:
# ── ABC-анализ по марже по каждому складу ──────────────────────────
if not has_cost:
    print('⚠️ Себестоимость отсутствует — ABC по марже по складам недоступен.')
else:
    for wh in warehouses:
        df_wh = df[df['Склад'] == wh]
        if len(df_wh) == 0:
            continue
        print(f'\n{"="*60}')
        print(f'💹 ABC-анализ по марже — склад: {wh}')
        print(f'{"="*60}')

        margin_agg_wh = df_wh.groupby('Блюдо').agg(
            Выручка=('Сумма со скидкой, р.', 'sum'),
            Маржа=('Маржа, р.', 'sum'),
            Продаж=('Блюдо', 'count'),
        ).reset_index()
        margin_agg_wh = margin_agg_wh.sort_values('Маржа', ascending=False).reset_index(drop=True)
        margin_agg_wh['Маржа, %'] = margin_agg_wh['Маржа'] / margin_agg_wh['Выручка'] * 100
        total_margin_wh = margin_agg_wh['Маржа'].sum()
        if total_margin_wh == 0:
            print(f'  ⚠️ Маржа по складу {wh} равна нулю — пропускаем.')
            continue
        margin_agg_wh['Доля маржи, %'] = margin_agg_wh['Маржа'] / total_margin_wh * 100
        margin_agg_wh['Накопл. доля маржи, %'] = margin_agg_wh['Доля маржи, %'].cumsum()

        def abc_m_wh(cumsum):
            if cumsum <= 80: return 'A'
            if cumsum <= 95: return 'B'
            return 'C'
        margin_agg_wh['ABC_маржа'] = margin_agg_wh['Накопл. доля маржи, %'].apply(abc_m_wh)
        abc_colors = {'A': PALETTE[1], 'B': PALETTE[2], 'C': PALETTE[3]}

        top_m_wh = margin_agg_wh.head(TOP_N)

        fig, axes = plt.subplots(1, 2, figsize=(18, max(6, TOP_N * 0.45)))
        fig.suptitle(f'💹 [{wh}] ABC-анализ по марже — топ-{TOP_N} блюд', fontsize=14, fontweight='bold')

        bar_colors_m = [abc_colors[c] for c in top_m_wh['ABC_маржа']]
        axes[0].barh(top_m_wh['Блюдо'][::-1], top_m_wh['Маржа'][::-1], color=bar_colors_m[::-1])
        for i, (v, cls) in enumerate(zip(top_m_wh['Маржа'][::-1], top_m_wh['ABC_маржа'][::-1])):
            axes[0].text(v * 0.01, i, f'{rub(v)}  [{cls}]', va='center', fontsize=8.5)
        axes[0].set_title('По сумме маржи (цвет = ABC-класс)')
        axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
        patches = [mpatches.Patch(color=abc_colors[c], label=f'Класс {c}') for c in 'ABC']
        axes[0].legend(handles=patches, frameon=False, fontsize=9)

        axes[1].barh(top_m_wh['Блюдо'][::-1], top_m_wh['Маржа, %'][::-1],
                     color=[PALETTE[1] if v >= 40 else PALETTE[3] for v in top_m_wh['Маржа, %'][::-1]])
        for i, v in enumerate(top_m_wh['Маржа, %'][::-1]):
            axes[1].text(v * 0.01, i, pct(v), va='center', fontsize=8.5)
        axes[1].axvline(40, color='gray', linestyle='--', alpha=0.6, label='Порог 40%')
        axes[1].set_title('Маржинальность, %')
        axes[1].legend(frameon=False, fontsize=9)

        plt.tight_layout()
        plt.show()

        abc_m_summary_wh = margin_agg_wh.groupby('ABC_маржа').agg(
            Блюд=('Блюдо', 'count'),
            Маржа=('Маржа', 'sum'),
        ).reset_index()
        abc_m_summary_wh['Доля маржи'] = (abc_m_summary_wh['Маржа'] / abc_m_summary_wh['Маржа'].sum() * 100).apply(pct)
        abc_m_summary_wh['Маржа'] = abc_m_summary_wh['Маржа'].apply(rub)
        print(f'\n📊 ABC-сводка по марже — {wh}:')
        print(abc_m_summary_wh.to_string(index=False))


## 📉 18. Себестоимость во времени <a id='cost_dynamics'></a>

Динамика себестоимости и маржинальности по неделям — позволяет отследить изменение закупочных цен или норм списания.


In [ ]:
if not has_cost:
    print('⚠️ Себестоимость отсутствует — раздел пропущен.')
else:
    # ── Динамика по неделям: выручка, себестоимость, маржа ──────────
    weekly_cost = df.groupby('Неделя').agg(
        Выручка=('Сумма со скидкой, р.', 'sum'),
        Себестоимость=(cost_col, 'sum'),
        Маржа=('Маржа, р.', 'sum'),
    ).reset_index()
    weekly_cost['Маржа, %'] = weekly_cost['Маржа'] / weekly_cost['Выручка'] * 100

    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
    fig.suptitle('📉 Динамика себестоимости и маржинальности по неделям',
                 fontsize=14, fontweight='bold')

    axes[0].bar(weekly_cost['Неделя'], weekly_cost['Выручка'],
                color=PALETTE[0], alpha=0.5, width=5, label='Выручка')
    axes[0].bar(weekly_cost['Неделя'], weekly_cost['Себестоимость'],
                color=PALETTE[3], alpha=0.6, width=5, label='Себестоимость')
    axes[0].plot(weekly_cost['Неделя'], weekly_cost['Маржа'],
                 color=PALETTE[1], linewidth=2.5, marker='o', markersize=4, label='Маржа')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
    axes[0].set_title('Выручка vs Себестоимость vs Маржа')
    axes[0].legend(frameon=False)

    axes[1].plot(weekly_cost['Неделя'], weekly_cost['Маржа, %'],
                 color=PALETTE[2], linewidth=2.5, marker='o', markersize=4)
    axes[1].fill_between(weekly_cost['Неделя'], weekly_cost['Маржа, %'],
                          alpha=0.15, color=PALETTE[2])
    axes[1].axhline(weekly_cost['Маржа, %'].mean(), color='gray', linestyle='--',
                    alpha=0.7, label=f'Среднее: {weekly_cost["Маржа, %"].mean():.1f}%')
    axes[1].set_title('Маржинальность, %')
    axes[1].set_ylabel('%')
    axes[1].legend(frameon=False)

    plt.tight_layout()
    plt.show()

    # ── Топ-10 блюд по себестоимости ────────────────────────────────
    cost_by_dish = df.groupby('Блюдо').agg(
        Выручка=('Сумма со скидкой, р.', 'sum'),
        Себестоимость=(cost_col, 'sum'),
        Маржа=('Маржа, р.', 'sum'),
        Продаж=('Блюдо', 'count'),
    ).reset_index()
    cost_by_dish['Маржа, %'] = cost_by_dish['Маржа'] / cost_by_dish['Выручка'] * 100
    cost_by_dish = cost_by_dish.sort_values('Себестоимость', ascending=False).head(TOP_N)

    fig2, ax2 = plt.subplots(figsize=(12, max(5, TOP_N * 0.5)))
    fig2.suptitle(f'💸 Топ-{TOP_N} блюд по себестоимости', fontsize=13, fontweight='bold')
    bar_cols_c = [PALETTE[1] if v >= 40 else PALETTE[3] for v in cost_by_dish['Маржа, %']]
    ax2.barh(cost_by_dish['Блюдо'][::-1], cost_by_dish['Себестоимость'][::-1], color=bar_cols_c[::-1])
    for i, (v, m) in enumerate(zip(cost_by_dish['Себестоимость'][::-1], cost_by_dish['Маржа, %'][::-1])):
        ax2.text(v * 0.01, i, f'{rub(v)}  (маржа {m:.1f}%)', va='center', fontsize=8.5)
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
    ax2.set_xlabel('Себестоимость, ₽')
    patches2 = [mpatches.Patch(color=PALETTE[1], label='Маржа ≥ 40%'),
                mpatches.Patch(color=PALETTE[3], label='Маржа < 40%')]
    ax2.legend(handles=patches2, frameon=False)
    plt.tight_layout()
    plt.show()

    print(f'\n📊 Средняя маржинальность за период: {pct(weekly_cost["Маржа, %"].mean())}')
    print(f'   Мин. маржинальность (неделя): {pct(weekly_cost["Маржа, %"].min())}')
    print(f'   Макс. маржинальность (неделя): {pct(weekly_cost["Маржа, %"].max())}')


### 🏭 Себестоимость во времени — по каждому складу

In [ ]:
# ── Себестоимость во времени по каждому складу ─────────────────────
if not has_cost:
    print('⚠️ Себестоимость отсутствует — раздел пропущен.')
else:
    for wh in warehouses:
        df_wh = df[df['Склад'] == wh]
        if len(df_wh) == 0:
            continue
        print(f'\n{"="*60}')
        print(f'📉 Себестоимость во времени — склад: {wh}')
        print(f'{"="*60}')

        weekly_cost_wh = df_wh.groupby('Неделя').agg(
            Выручка=('Сумма со скидкой, р.', 'sum'),
            Себестоимость=(cost_col, 'sum'),
            Маржа=('Маржа, р.', 'sum'),
        ).reset_index()
        if len(weekly_cost_wh) == 0:
            continue
        weekly_cost_wh['Маржа, %'] = weekly_cost_wh['Маржа'] / weekly_cost_wh['Выручка'] * 100

        fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
        fig.suptitle(f'📉 [{wh}] Динамика себестоимости и маржинальности по неделям',
                     fontsize=14, fontweight='bold')

        axes[0].bar(weekly_cost_wh['Неделя'], weekly_cost_wh['Выручка'],
                    color=PALETTE[0], alpha=0.5, width=5, label='Выручка')
        axes[0].bar(weekly_cost_wh['Неделя'], weekly_cost_wh['Себестоимость'],
                    color=PALETTE[3], alpha=0.6, width=5, label='Себестоимость')
        axes[0].plot(weekly_cost_wh['Неделя'], weekly_cost_wh['Маржа'],
                     color=PALETTE[1], linewidth=2.5, marker='o', markersize=4, label='Маржа')
        axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
        axes[0].set_title('Выручка vs Себестоимость vs Маржа')
        axes[0].legend(frameon=False)

        axes[1].plot(weekly_cost_wh['Неделя'], weekly_cost_wh['Маржа, %'],
                     color=PALETTE[2], linewidth=2.5, marker='o', markersize=4)
        axes[1].fill_between(weekly_cost_wh['Неделя'], weekly_cost_wh['Маржа, %'],
                              alpha=0.15, color=PALETTE[2])
        avg_margin_wh = weekly_cost_wh['Маржа, %'].mean()
        axes[1].axhline(avg_margin_wh, color='gray', linestyle='--',
                        alpha=0.7, label=f'Среднее: {avg_margin_wh:.1f}%')
        axes[1].set_title('Маржинальность, %')
        axes[1].set_ylabel('%')
        axes[1].legend(frameon=False)

        plt.tight_layout()
        plt.show()

        # Топ по себестоимости для склада
        cost_by_dish_wh = df_wh.groupby('Блюдо').agg(
            Выручка=('Сумма со скидкой, р.', 'sum'),
            Себестоимость=(cost_col, 'sum'),
            Маржа=('Маржа, р.', 'sum'),
            Продаж=('Блюдо', 'count'),
        ).reset_index()
        cost_by_dish_wh['Маржа, %'] = cost_by_dish_wh['Маржа'] / cost_by_dish_wh['Выручка'] * 100
        cost_by_dish_wh = cost_by_dish_wh.sort_values('Себестоимость', ascending=False).head(TOP_N)

        fig2, ax2 = plt.subplots(figsize=(12, max(5, TOP_N * 0.5)))
        fig2.suptitle(f'💸 [{wh}] Топ-{TOP_N} блюд по себестоимости', fontsize=13, fontweight='bold')
        bar_cols_c = [PALETTE[1] if v >= 40 else PALETTE[3] for v in cost_by_dish_wh['Маржа, %']]
        ax2.barh(cost_by_dish_wh['Блюдо'][::-1], cost_by_dish_wh['Себестоимость'][::-1], color=bar_cols_c[::-1])
        for i, (v, m) in enumerate(zip(cost_by_dish_wh['Себестоимость'][::-1], cost_by_dish_wh['Маржа, %'][::-1])):
            ax2.text(v * 0.01, i, f'{rub(v)}  (маржа {m:.1f}%)', va='center', fontsize=8.5)
        ax2.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rub))
        ax2.set_xlabel('Себестоимость, ₽')
        patches2 = [mpatches.Patch(color=PALETTE[1], label='Маржа >= 40%'),
                    mpatches.Patch(color=PALETTE[3], label='Маржа < 40%')]
        ax2.legend(handles=patches2, frameon=False)
        plt.tight_layout()
        plt.show()

        print(f'\n📊 [{wh}] Средняя маржинальность: {pct(weekly_cost_wh["Маржа, %"].mean())}')
        print(f'   Мин. маржинальность (неделя): {pct(weekly_cost_wh["Маржа, %"].min())}')
        print(f'   Макс. маржинальность (неделя): {pct(weekly_cost_wh["Маржа, %"].max())}')


## 📋 19. Итоговое резюме <a id='резюме'></a>

In [ ]:
best_wh   = wh_agg.iloc[0]['Склад']
best_dish = dish_agg_global.iloc[0]['Блюдо'] if len(dish_agg_global) > 0 else '—'

pairs_df_for_summary = pairs_df_global if pairs_df_global is not None and len(pairs_df_global) > 0 else pd.DataFrame()
top_pair  = f"{pairs_df_for_summary.iloc[0]['Товар A']} + {pairs_df_for_summary.iloc[0]['Товар B']}" if len(pairs_df_for_summary) > 0 else '—'
top_lift  = pairs_df_for_summary.iloc[0]['Лифт'] if len(pairs_df_for_summary) > 0 else 0

checks_all = df.groupby('check_key').agg(
    Сумма=('Сумма со скидкой, р.', 'sum'),
    Бонусы=('Тип скидки', lambda x: (x == 'Бонусы').any())
).reset_index()
avg_bonus_all    = checks_all[checks_all['Бонусы']]['Сумма'].mean() if checks_all['Бонусы'].any() else 0
avg_no_bonus_all = checks_all[~checks_all['Бонусы']]['Сумма'].mean() if (~checks_all['Бонусы']).any() else 0
bonus_uplift = (avg_bonus_all / avg_no_bonus_all - 1) * 100 if avg_no_bonus_all > 0 else 0

summary = f"""
╔══════════════════════════════════════════════════════════════════════╗
║                   ИТОГОВОЕ РЕЗЮМЕ ЗА ПЕРИОД                        ║
║              {_date_from}  —  {_date_to}                         ║
╠══════════════════════════════════════════════════════════════════════╣
║  Выручка:          {rub(total_rev):<20}                             ║
║  Чеков:            {n_checks:<20,}                                  ║
║  Средний чек:      {rub(avg_check):<20}                             ║
"""
if has_cost:
    summary += f"""║  Маржа:            {rub(total_margin):<20} ({pct(margin_pct)})         ║\n"""
summary += f"""
╠══════════════════════════════════════════════════════════════════════╣
║  Лучший склад:     {best_wh:<40}   ║
║  Топ-1 блюдо:      {best_dish:<40}   ║
║  Лучшая пара:      {top_pair:<40}   ║
║  Лифт пары:        {top_lift:<40}   ║
╠══════════════════════════════════════════════════════════════════════╣
║  Чеков с бонусами: {bonus_checks:<6,} ({pct(bonus_pct)})                          ║
║  Прирост чека      ║
║  с бонусами:       +{bonus_uplift:.1f}% к обычному чеку                     ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(summary)